## Set up a view over all quarters.
A view is a saved query. The data stays in the Parquet files, and ga now means “all quarters”. .as_posix() gives forward slashes, which DuckDB accepts on Windows.

In [ ]:
import duckdb, pathlib
RAW = pathlib.Path("data/raw/ga_sessions")
con = duckdb.connect()   # in-memory, nothing is copied
con.execute(f"CREATE VIEW ga AS SELECT * FROM read_parquet('{RAW.as_posix()}/*.parquet')")

## Schema
Check the real column names and types here before relying on any of the queries below.

In [ ]:
con.sql("DESCRIBE ga").df()

## A readable preview, selecting a few columns
Dot notation (totals.pageviews) reaches inside a struct. .df() gives a nicely formatted table in the notebook, and .show() truncates wide columns.

In [ ]:
con.sql("""
SELECT date, fullVisitorId, visitId,
       totals.pageviews, totals.transactions, totals.transactionRevenue,
       device.deviceCategory, trafficSource.medium, geoNetwork.country
FROM ga LIMIT 10
""").df()

## Look inside the hits array (one row per page or event)
unnest turns a session with 12 hits into 12 rows. Filtering to one day keeps it quick.

In [ ]:
con.sql("""
WITH h AS (
  SELECT fullVisitorId, visitId, unnest(hits) AS hit
  FROM ga WHERE date = '20161101'
)
SELECT fullVisitorId, visitId, hit.hitNumber, hit.type, hit.page.pagePath
FROM h ORDER BY fullVisitorId, visitId, hit.hitNumber LIMIT 20
""").df()

## Sessions and Conversions per day
In this dataset transactionRevenue is, as far as I know, in micros (millionths of a currency unit). Confirm that before using it, and confirm what totals.transactions means before building your label, as your project context requires.

For a visual browser instead of code, DBeaver (free) can open DuckDB and query Parquet files. That’s optional, and the notebook route does the job.

## Viewing in BigQuery
Billing is on for you, so be careful here. BigQuery charges by the columns it reads, and hits is the bulk of the data.

In [ ]:
import os
from google.cloud import bigquery
client = bigquery.Client(project=os.environ["GCP_PROJECT"])
t = client.get_table("bigquery-public-data.google_analytics_sample.ga_sessions_20161101")
print(t.num_rows, t.num_bytes)
client.list_rows(t, max_results=5).to_dataframe()

## Queries: a guarded helper that dry-runs first and caps spend
Selecting only date and totals here means BigQuery never reads hits, so it should process far less than the full table.
The cap_mb limit will refuse anything larger. For a single day, query the exact table (ga_sessions_20161101). For a range, filter with _TABLE_SUFFIX as in the download notebook.

In [ ]:
def bq(sql, cap_mb=200):
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    print(f"Will process {dry.total_bytes_processed/1024**2:.1f} MiB")
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=cap_mb * 1024**2)
    return client.query(sql, job_config=cfg).to_dataframe()

bq("""
SELECT date, COUNT(*) AS sessions,
       COUNTIF(totals.transactions IS NOT NULL) AS with_transactions
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_201611*`
GROUP BY date ORDER BY date
""")